# Adult Income Data Cleaning Lab

In [39]:
import sqlite3
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

raw_df = pd.read_csv('adult.csv')
raw_df.columns = raw_df.columns.str.strip()
df = raw_df.copy()

In [40]:
# Task 1: Hidden Missing Values
df.info()
display(df.head())

print('Before replacing ? values:')
display(df.isnull().sum())

df.replace(' ?', np.nan, inplace=True)

print('After replacing ? values:')
display(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education.num   32561 non-null  int64 
 5   marital.status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital.gain    32561 non-null  int64 
 11  capital.loss    32561 non-null  int64 
 12  hours.per.week  32561 non-null  int64 
 13  native.country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


Before replacing ? values:


,0
age,0
workclass,0
fnlwgt,0
education,0
education.num,0
marital.status,0
occupation,0
relationship,0
race,0
sex,0


After replacing ? values:


,0
age,0
workclass,0
fnlwgt,0
education,0
education.num,0
marital.status,0
occupation,0
relationship,0
race,0
sex,0


In [41]:
# Task 2: Missing Value Counts and Percentages
missing = pd.DataFrame({
    'count': df.isnull().sum(),
    'percentage': (df.isnull().mean() * 100).round(2)
}).sort_values('count', ascending=False)

display(missing)

,count,percentage
age,0,0.0
workclass,0,0.0
fnlwgt,0,0.0
education,0,0.0
education.num,0,0.0
marital.status,0,0.0
occupation,0,0.0
relationship,0,0.0
race,0,0.0
sex,0,0.0


Occupation has the most missing values (1,843), followed by workclass (1,836) and native-country (583).

In [42]:
# Task 3: Handling Missing Values
df[['workclass', 'occupation', 'native.country']] = df[
    ['workclass', 'occupation', 'native.country']
].fillna('Unknown')

print('Missing values left:', df.isnull().sum().sum())

Missing values left: 0


I used `Unknown` for the missing categorical values so the rows are not lost. There are no missing numeric values, so numeric imputation is not needed.

In [43]:
# Task 4: Duplicate Rows
duplicate_df = raw_df.replace(' ?', np.nan)

print('Exact duplicates:', duplicate_df.duplicated().sum())

feature_columns = ['age', 'workclass', 'fnlwgt', 'education',
                   'education.num', 'marital.status', 'occupation',
                   'relationship', 'race', 'sex', 'capital.gain',
                   'capital.loss', 'hours.per.week', 'native.country']
print('Duplicates without income:',
      duplicate_df.duplicated(subset=feature_columns).sum())

Exact duplicates: 24
Duplicates without income: 25


There are 24 exact duplicates and 25 duplicates when income is ignored. I did not remove the attribute-only duplicates because different people can have the same recorded attributes.

In [44]:
# Task 5: Categorical Values and Whitespace
for col in ['education', 'marital.status', 'native.country']:
    print(col, df[col].unique())

text_columns = ['workclass', 'education', 'marital.status', 'occupation',
                'relationship', 'race', 'sex', 'native.country', 'income']
for col in text_columns:
    df[col] = df[col].str.strip()

print('\nAfter stripping whitespace:')
for col in ['education', 'marital.status', 'native.country']:
    print(col, df[col].unique())

education ['HS-grad' 'Some-college' '7th-8th' '10th' 'Doctorate' 'Prof-school'
 'Bachelors' 'Masters' '11th' 'Assoc-acdm' 'Assoc-voc' '1st-4th' '5th-6th'
 '12th' '9th' 'Preschool']
marital.status ['Widowed' 'Divorced' 'Separated' 'Never-married' 'Married-civ-spouse'
 'Married-spouse-absent' 'Married-AF-spouse']
native.country ['United-States' '?' 'Mexico' 'Greece' 'Vietnam' 'China' 'Taiwan' 'India'
 'Philippines' 'Trinadad&Tobago' 'Canada' 'South' 'Holand-Netherlands'
 'Puerto-Rico' 'Poland' 'Iran' 'England' 'Germany' 'Italy' 'Japan' 'Hong'
 'Honduras' 'Cuba' 'Ireland' 'Cambodia' 'Peru' 'Nicaragua'
 'Dominican-Republic' 'Haiti' 'El-Salvador' 'Hungary' 'Columbia'
 'Guatemala' 'Jamaica' 'Ecuador' 'France' 'Yugoslavia' 'Scotland'
 'Portugal' 'Laos' 'Thailand' 'Outlying-US(Guam-USVI-etc)']

After stripping whitespace:
education ['HS-grad' 'Some-college' '7th-8th' '10th' 'Doctorate' 'Prof-school'
 'Bachelors' 'Masters' '11th' 'Assoc-acdm' 'Assoc-voc' '1st-4th' '5th-6th'
 '12th' '9th' 'Presc

In [45]:
# Task 6: Descriptive Statistics
display(df.describe())
display(df.describe(include='object'))

print('Mean age:', df['age'].mean())
print('Median age:', df['age'].median())
print('Mean hours per week:', df['hours.per.week'].mean())
print('Median hours per week:', df['hours.per.week'].median())

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


,workclass,education,marital.status,occupation,relationship,race,sex,native.country,income
count,32561,32561,32561,32561,32561,32561,32561,32561,32561
unique,9,16,7,15,6,5,2,42,2
top,Private,HS-grad,Married-civ-spouse,Prof-specialty,Husband,White,Male,United-States,<=50K
freq,22696,10501,14976,4140,13193,27816,21790,29170,24720


Mean age: 38.58164675532078
Median age: 37.0
Mean hours per week: 40.437455852092995
Median hours per week: 40.0


Mean and median age are 38.58 and 37. Mean and median hours-per-week are 40.44 and 40. The differences are small.

In [46]:
# Task 7: Frequency Analysis
occupation_counts = df['occupation'].value_counts()
print('Most common occupation:', occupation_counts.index[0])

print('\nSex percentage:')
display((df['sex'].value_counts(normalize=True) * 100).round(2))

print('Income percentage:')
display((df['income'].value_counts(normalize=True) * 100).round(2))

Most common occupation: Prof-specialty

Sex percentage:


,proportion
sex,
Male,66.92
Female,33.08


Income percentage:


,proportion
income,
<=50K,75.92
>50K,24.08


The income classes are imbalanced: 75.92% earn <=50K and 24.08% earn >50K.

In [47]:
# Task 8: Education and Education Number Check
education_check = df.groupby('education')['education.num'].unique()
display(education_check)

inconsistent = education_check[education_check.apply(len) > 1]
print('Number of inconsistencies:', len(inconsistent))

,education.num
education,
10th,[6]
11th,[7]
12th,[8]
1st-4th,[2]
5th-6th,[3]
7th-8th,[4]
9th,[5]
Assoc-acdm,[12]
Assoc-voc,[11]


Number of inconsistencies: 0


No inconsistencies were found.

In [48]:
# Task 9: SQL Data Extraction
connection = sqlite3.connect('adult_income.sqlite')
raw_df.to_sql('adult_income', connection, if_exists='replace', index=False)

sql_df = pd.read_sql_query('SELECT * FROM adult_income', connection)
over_30_df = pd.read_sql_query(
    'SELECT * FROM adult_income WHERE age > 30', connection
)
connection.close()

print('Original CSV shape:', raw_df.shape)
print('Full SQL result shape:', sql_df.shape)
print('Age > 30 result shape:', over_30_df.shape)
print('Shapes match:', raw_df.shape == sql_df.shape)
display(over_30_df.head())

Original CSV shape: (32561, 15)
Full SQL result shape: (32561, 15)
Age > 30 result shape: (21989, 15)
Shapes match: True


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [49]:
# Task 10: Summary

- The dataset has 32,561 rows and 15 columns and is used to predict whether income is above $50K.
- Missing values: occupation 1,843, workclass 1,836, and native-country 583.
- There are 24 exact duplicate rows. Categorical values also had extra whitespace.
- The most common occupation is Prof-specialty. The income label is imbalanced, with 75.92% in the <=50K class.
- Missing categorical values were changed to Unknown, whitespace was removed, and numeric columns did not need imputation.
- The full SQL result has the same shape as the original CSV: 32,561 rows and 15 columns.